# AstroCLIP Teaching Exercises

Complete the TODOs below to reproduce the full pipeline.

## 0. Environment Setup
Import the libraries, set `ASTROCLIP_ROOT`, and detect the device.

In [ ]:
# Imports and environment setup
import os
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from torch.utils.data import Dataset, DataLoader, TensorDataset
from datasets import load_from_disk
import umap

from astroclip.data.datamodule import AstroClipCollator
from teaching_scripts.data_utils import (
    build_image_dataloader,
    build_spectrum_dataloader,
    build_multimodal_dataloader,
)
from teaching_scripts.models import ImageAutoencoder, SpectrumAutoencoder, SmallCLIPModel

ASTROCLIP_ROOT = Path(os.environ.get("ASTROCLIP_ROOT", "/pbs/throng/training/astroinfo2025/data/AstroCLIP_data")).resolve()
ASTROCLIP_ROOT.mkdir(parents=True, exist_ok=True)

subset_path = ASTROCLIP_ROOT / "astroclip_subset_tiny"
if not subset_path.exists():
    raise FileNotFoundError(f"Dataset not found at {subset_path}. Prepare the subset first.")

ds = load_from_disk(subset_path)
print({split: len(ds[split]) for split in ds})

collator = AstroClipCollator(center_crop=144)

image_train_loader = build_image_dataloader(ds["train"], batch_size=128, shuffle=True, num_workers=0)
image_val_loader   = build_image_dataloader(ds["test"],  batch_size=128, shuffle=False, num_workers=0)

spectrum_train_loader = build_spectrum_dataloader(ds["train"], batch_size=256, shuffle=True, num_workers=0)
spectrum_val_loader   = build_spectrum_dataloader(ds["test"],  batch_size=256, shuffle=False, num_workers=0)

multimodal_train_loader = build_multimodal_dataloader(ds["train"], batch_size=256, shuffle=True, num_workers=0)
multimodal_val_loader   = build_multimodal_dataloader(ds["test"],  batch_size=256, shuffle=False, num_workers=0)

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print("Using device:", device)

plt.rcParams["figure.facecolor"] = "white"

def latest_version(log_dir: Path, run_name: str) -> Path:
    run_dir = log_dir / run_name
    candidates = sorted(run_dir.glob("version_*"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"No Lightning log directory found under {run_dir}")
    return candidates[-1]


## 0.a Visualize dataset
Create the image autoencoder and the new spectrum transformer.

In [ ]:
# Visualise multimodal examples (image + spectrum)
batch = next(iter(multimodal_val_loader))
n_examples = min(4, batch["image"].size(0))

fig, axes = plt.subplots(n_examples, 2, figsize=(10, 3 * n_examples))
for i in range(n_examples):
    img = batch["image"][i].cpu().permute(1, 2, 0).numpy()
    spec = batch["spectrum"][i].cpu().numpy()
    redshift = batch.get("redshift")
    targetid = batch.get("targetid")

    # next versions

    axes[i, 0].imshow(img)
    axes[i, 0].axis("off")
    title = "Multimodal sample"
    if redshift is not None:
        title += f" | z={float(redshift[i]):.3f}"
    if targetid is not None:
        title += f" | id={int(targetid[i])}"
    axes[i, 0].set_title(title)

    axes[i, 1].plot(spec, color="tab:blue", linewidth=1.0)
    axes[i, 1].set_xlabel("Pixel")
    axes[i, 1].set_ylabel("Flux")
    axes[i, 1].set_title("Spectrum")

plt.tight_layout()
plt.show()


## 1. Image & Spectrum Encoders
Create the image autoencoder and the new spectrum transformer.

In [ ]:
# Student-defined models (fill in the TODOs)

class StudentImageAutoencoder(L.LightningModule):
    def __init__(self, embed_dim=256, lr=1e-3, weight_decay=1e-5):
        super().__init__()
        self.save_hyperparameters()
        # TODO: replace the layers below with your own architecture
        self.encoder = ...
        self.decoder = ...

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

    def training_step(self, batch, batch_idx):
        images = batch
        recon = self(images)
        loss = F.mse_loss(recon, images)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images = batch
        recon = self(images)
        loss = F.mse_loss(recon, images)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


class StudentSpectrumAutoencoder(L.LightningModule):
    def __init__(self, input_dim=7781, embed_dim=256, lr=1e-3, weight_decay=1e-5):
        super().__init__()
        self.save_hyperparameters()
        # TODO: replace with a better 1-D architecture
        self.encoder = ...
        self.decoder = ...

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

    def training_step(self, batch, batch_idx):
        spectra = batch
        recon = self(spectra)
        loss = F.mse_loss(recon, spectra)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        spectra = batch
        recon = self(spectra)
        loss = F.mse_loss(recon, spectra)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


class StudentSpectrumTransformer(L.LightningModule):
    def __init__(self, input_dim=7781, patch_size=16, embed_dim=256, num_layers=2, num_heads=4, lr=1e-3, weight_decay=1e-5):
        super().__init__()
        self.save_hyperparameters()
        self.num_patches = math.ceil(input_dim / patch_size)
        self.pad = self.num_patches * patch_size - input_dim
        # TODO: define a patch embedding layer that maps (patch_size -> embed_dim)
        self.patch_embed = ...
        # Positional encoding is provided; no need to change unless you want to experiment
        self.positional_encoding = torch.nn.Parameter(torch.randn(1, self.num_patches, embed_dim))
        # TODO: create a Transformer encoder (encoder_layer + stack)
        encoder_layer = ...
        self.transformer = ...
        # TODO: normalisation + reconstruction back to the original spectrum length
        self.norm = ...
        self.reconstruction = ...

    def forward(self, spectrum: torch.Tensor):
        # TODO: implement patchify -> transformer -> pooling -> reconstruction
        raise NotImplementedError("Implement the spectrum transformer forward pass")

    def encode(self, spectrum: torch.Tensor):
        flat = spectrum.view(spectrum.size(0), -1)
        if self.pad > 0:
            flat = F.pad(flat, (0, self.pad))
        patches = flat.view(flat.size(0), self.num_patches, self.hparams.patch_size)
        tokens = self.patch_embed(patches) + self.positional_encoding
        encoded = self.transformer(tokens)
        return self.norm(encoded.mean(dim=1))

    def training_step(self, batch, batch_idx):
        spectra = batch
        recon = self(spectra)
        loss = F.mse_loss(recon, spectra)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        spectra = batch
        recon = self(spectra)
        loss = F.mse_loss(recon, spectra)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


class StudentCLIP(L.LightningModule):
    def __init__(self, image_encoder, spectrum_encoder, projection_dim=256, temperature=0.07, lr=5e-4, weight_decay=1e-5):
        super().__init__()
        self.save_hyperparameters(ignore=['image_encoder', 'spectrum_encoder'])
        self.image_encoder = image_encoder
        self.spectrum_encoder = spectrum_encoder
        # TODO: projection heads or other custom logic for the two modalities
        self.img_proj = ...
        self.spec_proj = ...
        self.temperature = temperature

    def forward(self, images, spectra):
        # TODO: encode images/spectra and return projected embeddings
        raise NotImplementedError("Implement the CLIP forward pass")

    def training_step(self, batch, batch_idx):
        images, spectra = batch['image'], batch['spectrum']
        img, spec = self(images, spectra)
        logits = (img @ spec.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, spectra = batch['image'], batch['spectrum']
        img, spec = self(images, spectra)
        logits = (img @ spec.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}


### Training or Loading
Train from scratch or load checkpoints for both encoders.

In [ ]:
# Train or load helper
def train_with_lightning(model, ckpt_path, train_loader, val_loader, *, max_epochs, logger_name):
    logger = CSVLogger(save_dir=str(ASTROCLIP_ROOT / "logs"), name=logger_name)
    checkpoint = ModelCheckpoint(
        dirpath=ckpt_path.parent,
        filename=ckpt_path.stem,
        save_last=True,
        save_top_k=1,
        monitor="val_loss",
        mode="min",
    )
    trainer = L.Trainer(
        accelerator="auto",
        devices="auto",
        max_epochs=max_epochs,
        log_every_n_steps=25,
        callbacks=[checkpoint],
        logger=logger,
    )
    trainer.fit(model, train_loader, val_loader)
    trainer.save_checkpoint(ckpt_path)
    return model

# Example usage:
spectrum_ckpt_path = ASTROCLIP_ROOT / "models/spectrum_transformer_student.ckpt"
if spectrum_ckpt_path.exists():
    spectrum_model = StudentSpectrumAutoencoder.load_from_checkpoint(spectrum_ckpt_path)
else:
    spectrum_model = StudentSpectrumAutoencoder()
    spectrum_model = train_with_lightning(
    spectrum_model,
    spectrum_ckpt_path,
    spectrum_train_loader,
    spectrum_val_loader,
    max_epochs=10,
    logger_name="spectrum_transformer_ex",
    )


image_ckpt_path = ASTROCLIP_ROOT / "models/image_autoencoder_student.ckpt"
if image_ckpt_path.exists():
    image_model = StudentImageTransformer.load_from_checkpoint(image_ckpt_path)
else:
    image_model = StudentImageAutoencoder()
    image_model = train_with_lightning(
    image_model,
    image_ckpt_path,
    image_train_loader,
    image_val_loader,
    max_epochs=1,
    logger_name="image_autoencoder_ex",
    )

In [ ]:
# Visualise image autoencoder reconstructions
image_model = image_model.to(device)
image_model.eval()

with torch.no_grad():
    images = next(iter(image_val_loader))
    inputs = images.to(device)
    recon = image_model(inputs).cpu()

n_show = min(6, inputs.size(0))
plt.figure(figsize=(12, 4))
for i in range(n_show):
    plt.subplot(2, n_show, i + 1)
    plt.imshow(images[i].permute(1, 2, 0).numpy())
    plt.axis("off")
    if i == 0:
        plt.title("Original")

    plt.subplot(2, n_show, n_show + i + 1)
    plt.imshow(recon[i].permute(1, 2, 0).numpy())
    plt.axis("off")
    if i == 0:
        plt.title("Reconstruction")
plt.tight_layout()
plt.show()

In [ ]:
# Visualise spectrum autoencoder reconstructions
spectrum_model = spectrum_model.to(device)
spectrum_model.eval()

with torch.no_grad():
    spectra = next(iter(spectrum_val_loader))
    inputs = spectra.to(device)
    recon = spectrum_model(inputs).cpu()

n_show = min(5, inputs.size(0))
plt.figure(figsize=(10, 6))
for i in range(n_show):
    plt.subplot(n_show, 1, i + 1)
    plt.plot(spectra[i].numpy(), label="original", alpha=0.7)
    plt.plot(recon[i].numpy(), label="reconstruction", alpha=0.7)
    plt.ylabel("Flux")
    if i == 0:
        plt.title("Spectrum Autoencoder Reconstructions")
    if i == n_show - 1:
        plt.xlabel("Pixel")
    if i == 0:
        plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 2. CLIP Alignment & Embedding Exploration
Train/Load CLIP model and run UMAP.

In [ ]:
class StudentCLIP(torch.nn.Module):
    def __init__(self, image_encoder, spectrum_encoder, projection_dim=256, temperature=0.07):
        super().__init__()
        self.image_encoder = image_encoder
        self.spectrum_encoder = spectrum_encoder
        # TODO: define projection layers (image -> projection_dim, spectrum -> projection_dim)
        self.img_proj = ...
        self.spec_proj = ...
        self.temperature = temperature

    def encode_image(self, images):
        # TODO: encode images, project, and L2-normalize
        raise NotImplementedError

    def encode_spectrum(self, spectra):
        # TODO: encode spectra, project, and L2-normalize
        raise NotImplementedError

    def forward(self, images, spectra):
        img = self.encode_image(images)
        spec = self.encode_spectrum(spectra)
        return img, spec

    def training_step(self, batch, batch_idx):
        images, spectra = batch['image'], batch['spectrum']
        img, spec = self(images, spectra)
        logits = (img @ spec.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, spectra = batch['image'], batch['spectrum']
        img, spec = self(images, spectra)
        logits = (img @ spec.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=5e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}


In [ ]:
# Generate embeddings for the validation split and visualise with UMAP
image_embeddings, spectrum_embeddings, redshift = [], [], []

clip_model.eval()
with torch.no_grad():
    for batch in multimodal_val_loader:
        imgs = batch["image"].to(device)
        specs = batch["spectrum"].to(device)
        img_embeds, spec_embeds = clip_model(imgs, specs)
        image_embeddings.append(img_embeds.cpu().numpy())
        spectrum_embeddings.append(spec_embeds.cpu().numpy())
        redshift.append(batch["redshift"].numpy())

image_embeddings = np.concatenate(image_embeddings)
spectrum_embeddings = np.concatenate(spectrum_embeddings)
redshift = np.concatenate(redshift)

reducer = umap.UMAP(random_state=42)
joint = reducer.fit_transform(np.concatenate([image_embeddings, spectrum_embeddings], axis=0))
labels = np.concatenate([np.zeros(len(image_embeddings)), np.ones(len(spectrum_embeddings))])

plt.figure(figsize=(6, 5))
plt.scatter(joint[labels == 0, 0], joint[labels == 0, 1], s=5, alpha=0.5, label="Images")
plt.scatter(joint[labels == 1, 0], joint[labels == 1, 1], s=5, alpha=0.5, label="Spectra")
plt.legend()
plt.title("UMAP of joint embedding space")
plt.show()


## 3. Regression Head
Use the embeddings to train a regression model.

In [ ]:
# Regression head on image embeddings
train_size = int(0.8 * len(image_embeddings))
train_emb = image_embeddings[:train_size]
val_emb = image_embeddings[train_size:]
train_z = redshift[:train_size]
val_z = redshift[train_size:]

train_ds = TensorDataset(torch.from_numpy(train_emb).float(), torch.from_numpy(train_z).float())
val_ds = TensorDataset(torch.from_numpy(val_emb).float(), torch.from_numpy(val_z).float())
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256, shuffle=False)

class StudentRegressor(torch.nn.Module):
    def __init__(self, input_dim, hidden=256):
        super().__init__()
        # TODO: build a small MLP that maps input_dim -> 1
        self.model = ...

    def forward(self, x):
        # TODO: forward through the MLP defined above
        raise NotImplementedError("Implement the regression head forward pass")

def train_regressor(model, train_loader, val_loader, epochs=40):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    criterion = torch.nn.MSELoss()
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)

        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                running += loss.item() * xb.size(0)
        val_loss = running / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:02d}: train={train_loss:.4f}, val={val_loss:.4f}")

    return train_losses, val_losses

# TODO: instantiate StudentRegressor, train it, and plot losses / predictions


## 4. CNN Baseline
Train a CNN directly on images for comparison.

In [ ]:
# Image-only CNN baseline
class ImageRedshiftDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        record = dict(self.dataset[idx])
        image = torch.tensor(np.asarray(record["image"]), dtype=torch.float32).permute(2, 0, 1)
        redshift = torch.tensor(float(record["redshift"]), dtype=torch.float32)
        return image, redshift

cnn_train_dl = DataLoader(ImageRedshiftDataset(ds["train"]), batch_size=128, shuffle=True, num_workers=0)
cnn_val_dl   = DataLoader(ImageRedshiftDataset(ds["test"]),  batch_size=128, shuffle=False, num_workers=0)

class StudentImageCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: design a compact CNN feature extractor and regression head
        self.feature_extractor = ...
        self.regressor = ...

    def forward(self, x):
        # TODO: implement the forward pass combining extractor + regressor
        raise NotImplementedError("Implement the CNN forward pass")

def train_cnn(model, train_loader, val_loader, epochs=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = torch.nn.MSELoss()
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)

        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                running += loss.item() * xb.size(0)
        val_loss = running / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        tqdm.write(f"Epoch {epoch+1:02d}: train={train_loss:.4f}, val={val_loss:.4f}")

    return train_losses, val_losses

# TODO: instantiate StudentImageCNN, train it with train_cnn, and plot the losses


## 5. Compare Results
Plot predicted vs. true redshift for CLIP and baseline; discuss.

In [ ]:
# Comparison cell
# TODO:
# 1. Collect predictions from your embedding regressor and CNN baseline.
# 2. Compute metrics (e.g., MSE / MAE) for both.
# 3. Plot predicted vs true redshift for each model side by side.
# 4. Summarise which approach performed better and why.
